# FrameWeave Hybrid v6.0 (V5.1) on Video-LLaVA-7B

Colab training notebook (L4 / A100). FrameWeave Hybrid v6.0 implements formulation **V5.1**. It does **not** append join tokens. It writes a gated RMS-calibrated residual into Video-LLaVA visual tokens after the frozen projector, then frozen Vicuna reads the same visual slice.

$$
\widetilde x_{ip} = x_{ip} + m_{ip}\,\chi_i\,a_{ip}\,c_{ip}\,u_{ip}
$$

- Clean pretrained Video-LLaVA, **frozen**. Hybrid is trained **joint-from-scratch** (no V3 weights, no GlobalTimeFix init checkpoint).
- $W_o=0$ (and bias $0$) at init → **exact Vanilla**. $P_L$ is small-Xavier (`gain=0.01`); Vanilla still holds because $u=W_o h=0$.
- $K=1$ remains identity because $\chi_i=0$ (pair mask excludes $i=j$)
- Question-guided frame pooling + non-periodic time $(\tau,\rho)$. Time lives in **Global only**.
- Local Q/K are **post-LayerNorm**. Local has **no** $A_t\phi_{ij}$.
- Backbone: `LanguageBind/Video-LLaVA-7B` (same pin as DTR / old RelJoin). Residual is in-place; sequence length unchanged.
- Loss: $\mathrm{logsumexp}(\mathcal L_{\mathrm{cand}},\mathcal L_{\mathrm{corr}})-\log 2$ on all samples. Vanilla scores are stopgrad. Warmup **10%**.
- Default train: `VARIANT="Hybrid"`, `TRAIN_STAGE="joint"`, `INIT_CKPT=""`. Ablations: `GlobalTimeFix`, `HybridNoTime`, `HybridNoGlobal`.
- Train data: PerceptionTest MC from LLaVA-Video-178K. HERBench is a separate eval notebook.
- Torch runs in `/content/dtr_env` (Python 3.11 + torch 2.0.1 + transformers 4.31.0, fp16). DTR repo is the Video-LLaVA loader.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

## `RUN_MODE`

`unit` | `smoke` | `overfit` | `lite`

Overfit gate (32 samples): V5.1 loss drop ≥ 30%, train acc ≥ 90%, no residual/grad explosion. Do not run `lite` until overfit passes.


## 1. Configuration

**Input:** none · **Output:** `RUN_MODE`, `VARIANT`, `TRAIN_STAGE`, pins, paths


In [ ]:
# 1. Configuration — change RUN_MODE / VARIANT / STAGE here
from __future__ import annotations
from pathlib import Path

RUN_MODE = "unit"  # unit | smoke | overfit | lite
TRAIN_STAGE = "joint"  # joint-from-scratch (alias A still accepted by CLI)
VARIANT = "Hybrid"  # default: Global+Local together
INIT_CKPT = ""  # empty for joint Hybrid; do not load a GlobalTimeFix checkpoint

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

MODEL_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"
DTR_REPO_URL = "https://github.com/ifwub234/DTR.git"
DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
TRAIN_DATASET_ID = "lmms-lab/LLaVA-Video-178K"
TRAIN_DATASET_REVISION = "6d8c562dc26d70042a0d9704d1cae58c94b89098"

VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
DTR_REPO = ROOT / "DTR"

print({
    "RUN_MODE": RUN_MODE,
    "TRAIN_STAGE": TRAIN_STAGE,
    "VARIANT": VARIANT,
    "INIT_CKPT": INIT_CKPT,
    "ckpt_name": f"{VARIANT}.pt",
    "ROOT": str(ROOT),
    "OUT_DIR": str(OUT_DIR),
    "method": "FrameWeave-Hybrid",
    "method_version": "6.0.0",
    "formulation": "5.1.0",
})


## 2. Environment installation (`dtr_env`)

Same bootstrap as the old RelJoin / DTR Colab notebooks.


In [ ]:
# 2. Bootstrap isolated dtr_env (Python 3.11 + torch 2.0.1 như DTR)
from __future__ import annotations
import os, sys, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "dtr_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=check)

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PIP), "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PIP), "install", "--disable-pip-version-check",
    "torch==2.0.1+cu118", "torchvision==0.15.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118",
])
sh([
    str(PIP), "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "hf-transfer", "numpy==1.26.4", "transformers==4.31.0", "tokenizers==0.13.3", "accelerate==0.21.0",
    "datasets==2.21.0", "huggingface_hub==0.24.7", "einops==0.7.0", "sentencepiece==0.2.0", "protobuf<5",
    "av==12.3.0", "pandas==2.2.2", "scipy==1.13.1", "tqdm==4.67.1", "opencv-python-headless==4.10.0.84",
    "Pillow==10.4.0", "pytorchvideo==0.1.5", "fvcore==0.1.5.post20221221", "iopath==0.1.10", "decord==0.6.0",
])
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, torchvision, transformers; "
    "print(torch.__version__); print(torchvision.__version__); print(transformers.__version__); "
    "print(torch.__file__); print(torchvision.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/dtr_env/" in line, f"Package leaked outside venv: {line}"
assert (VENV_BIN / "huggingface-cli").exists()
print("dtr_env ready:", PYTHON)


In [ ]:
!/content/dtr_env/bin/pip install --disable-pip-version-check "peft==0.4.0" "timm==0.6.13" "scikit-learn==1.3.2" "matplotlib==3.8.4"


## 3. Reproducibility — paths, Drive, pin DTR


In [ ]:
# 3. Paths, Drive, clone DTR
from __future__ import annotations
import hashlib, os, subprocess
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
DTR_REPO = ROOT / "DTR"
DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"

for p in [MODEL_DIR.parent, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not (DTR_REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/ifwub234/DTR.git", str(DTR_REPO)], check=True)
subprocess.run(["git", "-C", str(DTR_REPO), "fetch", "--all"], check=False)
subprocess.run(["git", "-C", str(DTR_REPO), "checkout", DTR_COMMIT], check=True)
got = sha256_file(DTR_REPO / "run_inference_dtr.py")
assert got == DTR_SOURCE_SHA256, f"DTR SHA mismatch: {got}"

print({
    "DTR_REPO": str(DTR_REPO), "DTR_COMMIT": DTR_COMMIT,
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
    "OUT_DIR": str(OUT_DIR),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "videollava"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave sources
from pathlib import Path
ROOT = Path("/content")
required = [
    ROOT / "run_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "videollava_hooks.py",
    ROOT / "reljoin" / "unit_tests.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "data_perception.py",
    ROOT / "reljoin" / "frame_io.py",
    ROOT / "reljoin" / "losses.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Dataset / model download (gated by `RUN_MODE`)


In [ ]:
# 5. Conditional downloads (no HERBench in this training notebook)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
assert PYTHON.exists()
os.environ["PATH"] = f"{ROOT / 'dtr_env' / 'bin'}:{os.environ.get('PATH','')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"

MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
MODEL_HF_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"

need_model = RUN_MODE != "unit"
need_perception = RUN_MODE in {"smoke", "overfit", "lite"}
print({"need_model": need_model, "need_perception": need_perception})

if need_model:
    print("Downloading Video-LLaVA-7B...")
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    py = (
        "import os\n"
        "os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'\n"
        "from huggingface_hub import snapshot_download\n"
        f"snapshot_download(repo_id='{MODEL_HF_ID}', revision='{MODEL_REVISION}', "
        f"local_dir=r'{MODEL_DIR}', ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])\n"
        "print('model OK')\n"
    )
    subprocess.run([str(PYTHON), "-c", py], check=True)
    assert (MODEL_DIR / "config.json").exists()

if need_perception:
    print("PerceptionTest shard fetched by run_reljoin.py on demand.")

print("Download step done for RUN_MODE=", RUN_MODE)


## 6. FrameWeave Hybrid v6.0 in `reljoin/`

Forward: `frames → frozen tower → frozen projector → X → FrameWeave v6 → X̃ (same shape) → frozen LLM`

Vanilla bypass writes nothing. LLM weights are frozen but the multimodal forward is **not** `no_grad`, so gradients reach $\widetilde X$ and the joining module.

Question vector $h_q$ is the last non-pad hidden of a frozen text-only LLM pass. Directed relations use $M_{ij}=f_i f_j\mathbf 1[i\neq j]$. Time is $(\tau,\rho)$ in Global only. $W_o=0$ at init is exact Vanilla.

## 7. Mathematical unit tests


In [ ]:
# 7. Unit tests via dtr_env
from __future__ import annotations
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
cmd = [str(PYTHON), str(ROOT / "run_reljoin.py"), "--run-mode", "unit", "--root", "/content"]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, cwd="/content")
if proc.returncode != 0:
    raise RuntimeError(f"unit tests failed ({proc.returncode}). Scroll up for traceback.")


## 8. External PerceptionTest training

_External uniform-frame warm-start — not definitive multi-evidence supervision._

HERBench never enters training / val / early stopping / checkpoint selection.

Cell 1 `VARIANT="Hybrid"` + `TRAIN_STAGE="joint"` writes `Hybrid.pt`. Ablations write `GlobalTimeFix.pt`, `HybridNoTime.pt`, `HybridNoGlobal.pt`.

`logs/train_metrics.csv` is overwritten in place. Overfit gate (30% V5.1 loss drop, train acc ≥ 90%) is for Hybrid joint-from-scratch.


In [ ]:
# 8. Train FrameWeave Hybrid (gated by RUN_MODE)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    VARIANT
except NameError:
    VARIANT = "Hybrid"
try:
    TRAIN_STAGE
except NameError:
    TRAIN_STAGE = "joint"
try:
    INIT_CKPT
except NameError:
    INIT_CKPT = ""

os.environ.setdefault("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_videollava_outputs")
os.environ.setdefault("RELJOIN_ROOT", "/content")

if RUN_MODE in {"smoke", "overfit", "lite"}:
    chk = subprocess.check_output(
        [str(PYTHON), "-c", "from reljoin.data_perception import DATA_PARSER_VERSION; print(DATA_PARSER_VERSION)"],
        cwd="/content",
        text=True,
    ).strip()
    print("DATA_PARSER_VERSION:", chk)
    assert chk == "perception_conversations_mc_v3", (
        f"Colab vẫn dùng parser cũ ({chk}). Restart runtime rồi Run All để pull repo."
    )

    cmd = [
        str(PYTHON), str(ROOT / "run_reljoin.py"),
        "--run-mode", RUN_MODE,
        "--root", "/content",
        "--variant", VARIANT,
        "--stage", TRAIN_STAGE,
        "--init-ckpt", INIT_CKPT,
    ]
    print("VARIANT:", VARIANT, "STAGE:", TRAIN_STAGE)
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd="/content")
    if proc.returncode != 0:
        raise RuntimeError(
            f"run_reljoin exited {proc.returncode}. "
            "Xem traceback NGAY PHÍA TRÊN (dòng Kept/missing_video/AssertionError). "
            "Nếu missing_video: kiểm tra đã giải nén videos_1.tar.gz chưa."
        )
    ckpt = ROOT / "outputs_reljoin_v6_videollava" / "checkpoints" / "best_external_val" / f"{VARIANT}.pt"
    print("Wrote", ckpt, "exists=", ckpt.is_file())
else:
    print(f"Skip training for RUN_MODE={RUN_MODE}")


## 9. Loss curves and overfit go/no-go


In [ ]:
# 9. Plot train_metrics.csv + overfit gate
from __future__ import annotations
from pathlib import Path
import json

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_videollava"
csv_path = OUT / "logs" / "train_metrics.csv"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    VARIANT
except NameError:
    VARIANT = "Hybrid"
try:
    SKIP_OVERFIT_GATE
except NameError:
    SKIP_OVERFIT_GATE = False

if not csv_path.exists():
    print("No train_metrics.csv yet (run smoke/overfit/lite).")
else:
    import pandas as pd
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    display(df.tail(20))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    axes[0].plot(df["epoch"], df["ans_nll"], marker="o", label="ans_nll")
    if "val_loss" in df and df["val_loss"].notna().any():
        axes[0].plot(df["epoch"], df["val_loss"], marker="x", label="val_loss")
    axes[0].set_title("NLL")
    axes[0].legend()
    axes[1].plot(df["epoch"], df["train_acc"], marker="o", label="train_acc")
    if "val_acc" in df and df["val_acc"].notna().any():
        axes[1].plot(df["epoch"], df["val_acc"], marker="x", label="val_acc")
    axes[1].set_title("letter probe acc")
    axes[1].legend()
    if "residual_rms" in df:
        axes[2].plot(df["epoch"], df["residual_rms"], marker="o", label="residual_rms")
        axes[2].plot(df["epoch"], df["grad_norm"], marker="x", label="grad_norm")
        axes[2].set_title("residual / grad")
        axes[2].legend()
    fig.tight_layout()
    fig.savefig(OUT / "logs" / "loss_curves.png", dpi=120)
    plt.show()

    first = float(df["ans_nll"].iloc[0])
    last = float(df["ans_nll"].iloc[-1])
    drop = (first - last) / first if first > 0 else 0.0
    acc = float(df["train_acc"].iloc[-1])
    gmax = float(df["grad_norm"].max()) if "grad_norm" in df else float("nan")
    rmax = float(df["residual_rms"].max()) if "residual_rms" in df else float("nan")
    explosion = (gmax > 1e3) or (rmax > 1e3)
    gate = {
        "nll_first": first,
        "nll_last": last,
        "nll_drop": drop,
        "train_acc_last": acc,
        "grad_norm_max": gmax,
        "residual_rms_max": rmax,
        "nll_drop_ge_30pct": drop >= 0.30,
        "train_acc_ge_90pct": acc >= 0.90,
        "no_explosion": not explosion,
    }
    gate["variant"] = VARIANT
    gate["variant"] = VARIANT
    gate["overfit_go"] = (
        RUN_MODE != "overfit"
        or SKIP_OVERFIT_GATE
        or (gate["nll_drop_ge_30pct"] and gate["train_acc_ge_90pct"] and gate["no_explosion"])
    )
    (OUT / "overfit_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
    print(json.dumps(gate, indent=2))
    if SKIP_OVERFIT_GATE:
        print("Skipping overfit gate for this variant.")
    elif RUN_MODE == "overfit" and not gate["overfit_go"]:
        print("NO-GO: do not start lite (448) until this gate passes.")
    elif RUN_MODE == "overfit":
        print("GO: overfit gate passed. You may set RUN_MODE='lite' and re-run training cells.")


## 10. Export artifacts

`copytree(..., dirs_exist_ok=True)` overwrites Drive persist in place (`train_metrics.csv`, zip). `Hybrid.pt` on Drive is replaced when this run trained Hybrid.


In [ ]:
# 10. Copy outputs to Drive persist dir + zip
from __future__ import annotations
import shutil
from pathlib import Path
from datetime import datetime

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_videollava"
persist = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")
if not persist.exists():
    persist = OUT
persist.mkdir(parents=True, exist_ok=True)
if OUT.exists() and persist.resolve() != OUT.resolve():
    shutil.copytree(OUT, persist, dirs_exist_ok=True)
    print("Copied", OUT, "→", persist)

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_v6_videollava_{stamp}"
if OUT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
